# Drawing QASP quantum circuits

This notebook contains drawing-only examples for QASP policies, quantum kernels, and directly constructed circuits. Each example exports publication-ready Quantikz/TikZ and renders it inside Jupyter when a LaTeX installation is available.


## Requirements

Install `qasp[preview]` in the notebook kernel. For inline TikZ figures, install pdfLaTeX with Quantikz and standalone, and Ghostscript. Restart the kernel after upgrading QASP.

In [ ]:
import copy

import numpy as np
import torch

from qasp import (
    AngleAnsatz,
    AngleEmbedding,
    CircuitBuilder,
    create_kernel,
    create_policy,
    draw_kernel,
    draw_policy,
    show_tikz,
)

print("Torch:", torch.__version__)

In [ ]:
PASTEL_COLORS = {
    "single": "DCECF8",       # light blue
    "two": "E4D8F2",          # soft purple
    "multi": "EEE2F7",        # pale violet
    "measurement": "E8E5F5", # light lavender
}

FULL_TIKZ = {
    "standalone": True,
    "symbolic_parameters": True,
    "gate_colors": PASTEL_COLORS,
}

COMPRESSED_TIKZ = {
    **FULL_TIKZ,
    "compress": True,
    "max_explicit_layers": 4,
}


## 1. IQP policy

The first drawing shows every gate. The second keeps the complete first variational layer inside a dashed region and replaces the later layers with full-register unitary boxes.


In [ ]:
N_QUBITS = 4
N_FEATURES = 4
zero_input = torch.zeros(N_FEATURES)

policy_common = {
    "feature_map": "AngleEmbedding",
    "feature_map_config": {
        "features": N_FEATURES,
        "rotations": "Y",
        "imprimitive_pattern": "chain",
    },
    "n_actions": 2,
    "init": "zeros",
    "policy": "parity-like",
    "device": "default.qubit",
    "diff_method": "best",
    "softmax_activation": False,
    "output_scaling": None,
    "training_method": "expectation",
    "torchrl": False,
    "parity_cnot_decomposition": False,
}

iqp_policy_config = {
    **copy.deepcopy(policy_common),
    "ansatz": "IQPAnsatz",
    "ansatz_config": {
        "n_layers": 3,
        "imprimitive_pattern": "ring",
        "reuploading": False,
    },
}

iqp_policy = create_policy("PQCBorn", iqp_policy_config, n_qubits=N_QUBITS)


In [ ]:
draw_policy(
    model=iqp_policy,
    inputs=zero_input,
    output="tikz", return_tex=False,
    tikz_options=FULL_TIKZ,
)


In [ ]:
draw_policy(
    model=iqp_policy,
    inputs=zero_input,
    output="tikz", return_tex=False,
    tikz_options=COMPRESSED_TIKZ,
)


## 2. Policy with an arbitrary AngleAnsatz

Here the variational layer mixes one- and two-qubit Pauli rotations. In particular, the `ZZ` rotations are rendered as symmetric two-qubit gates rather than controlled rotations.


In [ ]:
angle_policy_config = {
    **copy.deepcopy(policy_common),
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 3,
        "rotations": [
            "ZZ 0 1",
            "ZZ 1 2",
            "ZZ 2 3",
            "Z 0",
            "Y 1",
            "X 2",
        ],
        "reuploading": False,
        "include_imprimitive": False,
        "imprimitive": "CZ",
        "imprimitive_pattern": "chain",
        "include_final_imprimitive": False,
    },
}

angle_policy = create_policy(
    "PQCBorn", angle_policy_config, n_qubits=N_QUBITS
)


In [ ]:
draw_policy(
    model=angle_policy,
    inputs=zero_input,
    output="tikz", return_tex=False,
    tikz_options=FULL_TIKZ,
)

draw_policy(
    model=angle_policy,
    inputs=zero_input,
    output="tikz", return_tex=False,
    tikz_options=COMPRESSED_TIKZ,
)


## 3. IQP quantum kernels

Fidelity and local-fidelity drawings take two samples, `x` and `y`. The two halves share all angle and scale parameters. Only the encoded sample changes. Their compressed diagrams include the forward unitary blocks and the matching adjoint blocks in reverse order. A projected kernel only takes `x`.


In [ ]:
x = np.array([0.10, 0.20, 0.30, 0.40])
y = np.array([0.40, 0.30, 0.20, 0.10])

kernel_common = {
    "feature_map": "AngleEmbedding",
    "feature_map_config": {
        "features": N_FEATURES,
        "rotations": "Y",
        "imprimitive_pattern": "chain",
    },
    "ansatz": "IQPAnsatz",
    "ansatz_config": {
        "n_layers": 3,
        "imprimitive_pattern": "ring",
        "reuploading": False,
    },
    "init": "zeros",
    "device": "default.qubit",
}


### 3.1 Global fidelity kernel


In [ ]:
fidelity_kernel = create_kernel(
    {**copy.deepcopy(kernel_common), "kernel_type": "fidelity"},
    n_qubits=N_QUBITS,
)

draw_kernel(
    kernel=fidelity_kernel,
    x=x,
    y=y,
    output="tikz", return_tex=False,
    tikz_options=FULL_TIKZ,
)

draw_kernel(
    kernel=fidelity_kernel,
    x=x,
    y=y,
    output="tikz", return_tex=False,
    tikz_options=COMPRESSED_TIKZ,
)


### 3.2 Local-fidelity kernel with a full Pauli-Z string

The observable `ZZZZ 0 1 2 3` produces a measurement box on every qubit.


In [ ]:
local_config = {
    **copy.deepcopy(kernel_common),
    "kernel_type": "fidelity",
    "projectors": ["P 0", "P 1", "P 2", "P 3"],
}
local_kernel = create_kernel(local_config, n_qubits=N_QUBITS)

draw_kernel(
    kernel=local_kernel,
    x=x,
    y=y,
    output="tikz", return_tex=False,
    tikz_options=COMPRESSED_TIKZ,
)


### 3.3 Projected quantum kernel


In [ ]:
projected_config = {
    **copy.deepcopy(kernel_common),
    "kernel_type": "projected",
    "observables": ["Z 0", "Z 1", "Z 2", "Z 3"],
    "gamma": 0.5,
}
projected_kernel = create_kernel(projected_config, n_qubits=N_QUBITS)

draw_kernel(
    kernel=projected_kernel,
    x=x,
    output="tikz", return_tex=False,
    tikz_options=COMPRESSED_TIKZ,
)


## 4. Circuit assembled directly with QASP

This example bypasses the policy and kernel factories. It combines a custom multi-Pauli feature map, a repeated AngleAnsatz, and a tensor-product Pauli-Z measurement.


In [ ]:
custom_feature_map = AngleEmbedding(
    n_qubits=N_QUBITS,
    features=3,
    rotations=["Y 0", "ZZ 0 1", "YYY 1 2 3"],
)
custom_ansatz = AngleAnsatz(
    n_qubits=N_QUBITS,
    n_layers=3,
    rotations=["X 0", "YY 1 2", "ZZ 2 3"],
    feature_map=custom_feature_map,
    reuploading=False,
)

builder = CircuitBuilder(n_qubits=N_QUBITS)
builder.set_ansatz(custom_ansatz)
builder.set_measurement(
    "expectation",
    wires=range(N_QUBITS),
    observables=["ZZZZ 0 1 2 3"],
)
builder.build()

custom_inputs = torch.zeros(3)
custom_params = torch.zeros(custom_ansatz.get_param_shapes()[0])
custom_tikz = builder.to_tikz(
    inputs=custom_inputs,
    params=custom_params,
    **COMPRESSED_TIKZ,
)
show_tikz(custom_tikz)


## 5. PQDQN quantum Q-function

The same drawing API supports a Q-function with a four-layer AngleAnsatz,
trainable Y encoding, data reuploading, and a CNOT ring after every layer.
`draw_qfunction` accepts the same TikZ options as `draw_policy` and `draw_kernel`.
In the compact view, every U_l block includes encoding and variational gates.
Θ_l includes rotation and scaling parameters; the model encodes f = tanh(s).
Meters show local measurement bases; the Pauli products and classical readout
are specified by the model configuration.
This example uses parity `preset_actions=4`: three individual Z-product
features feed an affine layer with two Q-value outputs. The classical
weights and bias are outside the quantum circuit drawing.


In [ ]:
from qasp import create_qfunction, draw_qfunction

qfunction = create_qfunction({
    "ansatz": "AngleAnsatz",
    "ansatz_config": {"n_layers": 4, "rotations": "YZ", "reuploading": True,
                      "include_imprimitive": True, "imprimitive": "CNOT",
                      "imprimitive_pattern": "ring", "include_final_imprimitive": True},
    "feature_map": "AngleEmbedding",
    "feature_map_config": {"features": 4, "rotations": "Y", "input_scaling": True},
    "observables": "parity-like", "preset_actions": 4, "fc_layer": True, "fc_bias": True,
    "device": "default.qubit", "diff_method": "backprop",
}, n_qubits=4, n_actions=2)
draw_qfunction(model=qfunction, output="tikz", return_tex=False, tikz_options=FULL_TIKZ)
draw_qfunction(model=qfunction, output="tikz", return_tex=False, tikz_options=COMPRESSED_TIKZ)
# Equivalent entry points: qfunction.draw(...) and draw_policy(model=qfunction, ...).


## Notes

- Use `return_tex=False` for just the preview. To keep editable LaTeX, omit that argument and assign the result to `tex`. For an explicit export, write the standalone source to a file.
- Set `standalone=False` when you want only the snippet to paste into an existing paper or Beamer presentation.
- Set `gate_colors=False` for monochrome output.
- `AngleEmbedding` is expanded into rotations labelled `f_i`; input scaling is shown as `f_i \lambda_i`.
- Increase `max_explicit_layers` to show more unitary blocks before QASP inserts an ellipsis.
- Automatic compression currently supports `AngleAnsatz` and `IQPAnsatz` with or without data reuploading (encoding must precede the first layer); mirrored overlap compression still requires `reuploading=False`.
